# DiffSoup — Stochastic Differentiable Triangle (Steps 1–6)

Step 1: 1 ray / thread Möller–Trumbore intersection on GPU.
Step 2: smooth opacity α = sigmoid(signed edge distance / σ), deterministic.
Step 3: stochastic Bernoulli(α) decision via Philox RNG; mean over samples → α.
Steps 4–6: backward pass (dL/dV via atomicAdd), finite-difference gradient check,
and a race / determinism check on the float32 atomics.
All validated against the CPU reference.

**Runtime → Change runtime type → GPU** (T4 is fine).
Then run the cells top-to-bottom.


## 1. Environment check (installs ninja — required by torch JIT load)

In [ ]:
!pip install -q ninja
import sys, torch
print('python :', sys.version.split()[0])
print('torch  :', torch.__version__)
print('cuda?  :', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU detected — set Runtime → GPU and re-run.'
print('device :', torch.cuda.get_device_name(0))
!ninja --version
!nvcc --version | tail -n2


## 2. Drop source files into the Colab working dir

In [ ]:
%%writefile step0_cpu_reference.py
"""
Step 0 — CPU reference for ray-triangle intersection (Möller–Trumbore).

Used as ground truth for the CUDA kernel in later steps.

Convention:
  - Triangle V0, V1, V2 (CCW from +z looking down at z=0 plane)
  - Ray:  P(t) = origin + t * direction,  t > 0
  - Barycentric returned as (w, u, v) where  P = w*V0 + u*V1 + v*V2,  w = 1 - u - v
  - `u` corresponds to edge V0→V1, `v` to edge V0→V2
  - Distance to nearest edge in barycentric space = min(w, u, v)
"""

from __future__ import annotations

import numpy as np

EPS = 1e-8


def ray_triangle_intersect(
    origins: np.ndarray,        # (N, 3)
    directions: np.ndarray,     # (N, 3)
    v0: np.ndarray,             # (3,)
    v1: np.ndarray,             # (3,)
    v2: np.ndarray,             # (3,)
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Vectorized Möller–Trumbore.

    Returns:
      hit:   (N,) bool       — True if ray hits the triangle at t > EPS
      t:     (N,) float      — parametric distance along the ray (NaN if miss)
      bary:  (N, 3) float    — (w, u, v) barycentric coordinates (NaN if miss)
    """
    origins = np.asarray(origins, dtype=np.float64)
    directions = np.asarray(directions, dtype=np.float64)

    e1 = v1 - v0                          # (3,)
    e2 = v2 - v0                          # (3,)

    pvec = np.cross(directions, e2)       # (N, 3)
    det = pvec @ e1                       # (N,)

    # Parallel rays (det ≈ 0) — no unique intersection
    parallel = np.abs(det) < EPS
    inv_det = np.where(parallel, 0.0, 1.0 / np.where(parallel, 1.0, det))

    tvec = origins - v0                   # (N, 3)
    u = np.einsum("ij,ij->i", tvec, pvec) * inv_det

    qvec = np.cross(tvec, e1)             # (N, 3)
    v = np.einsum("ij,ij->i", directions, qvec) * inv_det

    t = (qvec @ e2) * inv_det
    w = 1.0 - u - v

    hit = (~parallel) & (u >= 0.0) & (v >= 0.0) & (w >= 0.0) & (t > EPS)

    nan = np.full_like(t, np.nan)
    t_out = np.where(hit, t, nan)
    bary = np.stack([w, u, v], axis=-1)
    bary_out = np.where(hit[:, None], bary, np.nan)

    return hit, t_out, bary_out


def edge_distance_bary(bary: np.ndarray) -> np.ndarray:
    """
    Per-ray distance to the nearest triangle edge in *barycentric* units.
    A value of 0 means the hit is exactly on an edge; 1/3 is the centroid.
    NaN propagates for missed rays.
    """
    return np.min(bary, axis=-1)


# ---------------------------------------------------------------------------
# Hand-verifiable tests
# ---------------------------------------------------------------------------

def _fmt(x: np.ndarray) -> str:
    if np.all(np.isnan(x)):
        return "—"
    return "[" + ", ".join(f"{c:+.4f}" if not np.isnan(c) else "  nan " for c in x) + "]"


def run_tests() -> None:
    V0 = np.array([0.0, 0.0, 0.0])
    V1 = np.array([1.0, 0.0, 0.0])
    V2 = np.array([0.0, 1.0, 0.0])

    # Each test: (name, origin, direction, expected_hit, expected_bary_or_None)
    # All rays here shoot in -z; the triangle lies on z = 0.
    cases = [
        ("centroid hit",            [1/3, 1/3, +1.0], [0, 0, -1], True,  [1/3, 1/3, 1/3]),
        ("near vertex V0",          [0.0, 0.0, +1.0], [0, 0, -1], True,  [1.0, 0.0, 0.0]),
        ("near vertex V1",          [1.0, 0.0, +1.0], [0, 0, -1], True,  [0.0, 1.0, 0.0]),
        ("near vertex V2",          [0.0, 1.0, +1.0], [0, 0, -1], True,  [0.0, 0.0, 1.0]),
        ("edge midpoint V1V2",      [0.5, 0.5, +1.0], [0, 0, -1], True,  [0.0, 0.5, 0.5]),
        ("just inside edge V1V2",   [0.495, 0.495, +1.0], [0, 0, -1], True, [0.010, 0.495, 0.495]),
        ("just outside edge V1V2",  [0.51, 0.51, +1.0], [0, 0, -1], False, None),
        ("outside (x<0)",           [-0.01, 0.5, +1.0], [0, 0, -1], False, None),
        ("outside (y<0)",           [0.5, -0.01, +1.0], [0, 0, -1], False, None),
        ("parallel to plane",       [0.3, 0.3, +1.0], [1, 0,  0], False, None),
        ("ray points away",         [0.3, 0.3, +1.0], [0, 0, +1], False, None),
        ("oblique hit (centroid)",  [1/3 + 1.0, 1/3 + 1.0, +1.0], [-1, -1, -1], True, [1/3, 1/3, 1/3]),
    ]

    origins = np.array([c[1] for c in cases], dtype=np.float64)
    dirs    = np.array([c[2] for c in cases], dtype=np.float64)

    hit, t, bary = ray_triangle_intersect(origins, dirs, V0, V1, V2)
    edge_d = edge_distance_bary(bary)

    print(f"Triangle: V0={V0.tolist()}  V1={V1.tolist()}  V2={V2.tolist()}\n")
    header = f"{'#':>2}  {'case':<26} {'hit':>5}  {'t':>8}  {'bary (w,u,v)':<28} {'min(bary)':>10}  result"
    print(header)
    print("-" * len(header))

    all_ok = True
    for i, (name, _, _, exp_hit, exp_bary) in enumerate(cases):
        ok = bool(hit[i]) == exp_hit
        if exp_hit and exp_bary is not None:
            ok = ok and np.allclose(bary[i], exp_bary, atol=1e-6)

        t_str = f"{t[i]:8.4f}" if hit[i] else "   —    "
        ed_str = f"{edge_d[i]:+.4f}" if hit[i] else "   —   "
        flag = "OK " if ok else "FAIL"
        all_ok = all_ok and ok
        print(f"{i:>2}  {name:<26} {str(bool(hit[i])):>5}  {t_str}  {_fmt(bary[i]):<28} {ed_str:>10}  {flag}")

    print()
    print("ALL TESTS PASSED" if all_ok else "SOME TESTS FAILED")


# ---------------------------------------------------------------------------
# Sanity check: batched edge-biased sampling (preview for step 1+)
# ---------------------------------------------------------------------------

def edge_biased_sample(n: int, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """
    Generate N rays that are biased toward the triangle's edges, all shooting in -z.
    This previews the sampling distribution used by later CUDA tests (N = 10^6).
    """
    rng = np.random.default_rng(seed)
    # Sample (u, v) uniformly on the triangle, then nudge toward the closest edge
    r1 = np.sqrt(rng.random(n))
    r2 = rng.random(n)
    u = 1.0 - r1
    v = r1 * (1.0 - r2)
    w = r1 * r2
    bary = np.stack([w, u, v], axis=-1)
    # Pull each point a fraction toward its nearest edge (set the smallest coord toward 0)
    idx = np.argmin(bary, axis=-1)
    pull = 0.5  # 0 = no bias, 1 = collapse onto the edge
    bary[np.arange(n), idx] *= (1.0 - pull)
    bary /= bary.sum(axis=-1, keepdims=True)

    V0 = np.array([0.0, 0.0, 0.0])
    V1 = np.array([1.0, 0.0, 0.0])
    V2 = np.array([0.0, 1.0, 0.0])
    points = bary[:, 0:1] * V0 + bary[:, 1:2] * V1 + bary[:, 2:3] * V2  # (n, 3)

    origins = points + np.array([0.0, 0.0, 1.0])
    directions = np.tile(np.array([0.0, 0.0, -1.0]), (n, 1))
    return origins, directions


def run_sampling_preview(n: int = 100_000) -> None:
    V0 = np.array([0.0, 0.0, 0.0])
    V1 = np.array([1.0, 0.0, 0.0])
    V2 = np.array([0.0, 1.0, 0.0])
    origins, dirs = edge_biased_sample(n)
    hit, _, bary = ray_triangle_intersect(origins, dirs, V0, V1, V2)
    hit_rate = hit.mean()
    edge_d = edge_distance_bary(bary[hit])
    print(f"\nedge-biased sampling preview (N={n}):")
    print(f"  hit rate         : {hit_rate:.4f}  (expect ~1.0; rays are constructed to hit)")
    print(f"  mean min(bary)   : {edge_d.mean():.5f}  (smaller = closer to edges)")
    print(f"  median min(bary) : {np.median(edge_d):.5f}")
    print(f"  fraction within 0.02 of an edge : {(edge_d < 0.02).mean():.4f}")


if __name__ == "__main__":
    run_tests()
    run_sampling_preview()

In [ ]:
%%writefile step1_cuda_forward.cu
// Step 1 — Deterministic ray-triangle intersection on CUDA (Möller–Trumbore).
// 1 ray per thread. No stochastic masking yet (that lands in step 3).
// Output convention matches step0_cpu_reference.py:
//   bary = (w, u, v) with P = w*V0 + u*V1 + v*V2, w = 1 - u - v
//   miss -> t = NaN, bary = NaN, hit = false

#include <torch/extension.h>
#include <cuda_runtime.h>
#include <vector>

template <typename scalar_t>
__global__ void intersect_kernel(
    const scalar_t* __restrict__ origins,     // (N, 3) row-major
    const scalar_t* __restrict__ directions,  // (N, 3) row-major
    const scalar_t* __restrict__ verts,       // 9 floats: v0xyz, v1xyz, v2xyz
    bool*           __restrict__ hit,         // (N,)
    scalar_t*       __restrict__ t_out,       // (N,)
    scalar_t*       __restrict__ bary,        // (N, 3)
    int N)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    const scalar_t EPS = (scalar_t)1e-8;
    const scalar_t NaN = (scalar_t)NAN;

    const scalar_t v0x = verts[0], v0y = verts[1], v0z = verts[2];
    const scalar_t v1x = verts[3], v1y = verts[4], v1z = verts[5];
    const scalar_t v2x = verts[6], v2y = verts[7], v2z = verts[8];

    const scalar_t ox = origins[3*i+0], oy = origins[3*i+1], oz = origins[3*i+2];
    const scalar_t dx = directions[3*i+0], dy = directions[3*i+1], dz = directions[3*i+2];

    const scalar_t e1x = v1x - v0x, e1y = v1y - v0y, e1z = v1z - v0z;
    const scalar_t e2x = v2x - v0x, e2y = v2y - v0y, e2z = v2z - v0z;

    // pvec = direction × e2
    const scalar_t px = dy*e2z - dz*e2y;
    const scalar_t py = dz*e2x - dx*e2z;
    const scalar_t pz = dx*e2y - dy*e2x;

    const scalar_t det = px*e1x + py*e1y + pz*e1z;
    const bool parallel = (det > -EPS) && (det < EPS);
    const scalar_t inv_det = parallel ? (scalar_t)0 : (scalar_t)1 / det;

    // tvec = origin - v0
    const scalar_t tx = ox - v0x, ty = oy - v0y, tz = oz - v0z;
    const scalar_t u = (tx*px + ty*py + tz*pz) * inv_det;

    // qvec = tvec × e1
    const scalar_t qx = ty*e1z - tz*e1y;
    const scalar_t qy = tz*e1x - tx*e1z;
    const scalar_t qz = tx*e1y - ty*e1x;

    const scalar_t v = (dx*qx + dy*qy + dz*qz) * inv_det;
    const scalar_t t = (qx*e2x + qy*e2y + qz*e2z) * inv_det;
    const scalar_t w = (scalar_t)1 - u - v;

    const bool h = (!parallel) && (u >= 0) && (v >= 0) && (w >= 0) && (t > EPS);

    hit[i] = h;
    t_out[i]      = h ? t : NaN;
    bary[3*i+0]   = h ? w : NaN;
    bary[3*i+1]   = h ? u : NaN;
    bary[3*i+2]   = h ? v : NaN;
}

std::vector<torch::Tensor> intersect_forward(
    torch::Tensor origins,
    torch::Tensor directions,
    torch::Tensor v0,
    torch::Tensor v1,
    torch::Tensor v2)
{
    TORCH_CHECK(origins.is_cuda(),    "origins must be CUDA");
    TORCH_CHECK(directions.is_cuda(), "directions must be CUDA");
    TORCH_CHECK(origins.dim() == 2 && origins.size(1) == 3,    "origins must be (N, 3)");
    TORCH_CHECK(directions.sizes() == origins.sizes(),         "directions must match origins");
    TORCH_CHECK(origins.scalar_type() == directions.scalar_type(), "dtype mismatch");
    TORCH_CHECK(v0.numel() == 3 && v1.numel() == 3 && v2.numel() == 3, "vertices must have 3 elements");

    origins    = origins.contiguous();
    directions = directions.contiguous();

    const auto N = origins.size(0);
    const auto opts = origins.options();

    // Pack the 3 vertices into a single 9-element device buffer in the same dtype.
    auto verts = torch::empty({9}, opts);
    verts.narrow(0, 0, 3).copy_(v0.to(opts.device()).to(opts.dtype()).view({3}));
    verts.narrow(0, 3, 3).copy_(v1.to(opts.device()).to(opts.dtype()).view({3}));
    verts.narrow(0, 6, 3).copy_(v2.to(opts.device()).to(opts.dtype()).view({3}));

    auto hit  = torch::empty({N},    opts.dtype(torch::kBool));
    auto t    = torch::empty({N},    opts);
    auto bary = torch::empty({N, 3}, opts);

    const int threads = 256;
    const int blocks  = (int)((N + threads - 1) / threads);

    AT_DISPATCH_FLOATING_TYPES(origins.scalar_type(), "intersect_kernel", [&] {
        intersect_kernel<scalar_t><<<blocks, threads>>>(
            origins.data_ptr<scalar_t>(),
            directions.data_ptr<scalar_t>(),
            verts.data_ptr<scalar_t>(),
            hit.data_ptr<bool>(),
            t.data_ptr<scalar_t>(),
            bary.data_ptr<scalar_t>(),
            (int)N);
    });

    return {hit, t, bary};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("intersect_forward", &intersect_forward,
          "Deterministic ray-triangle intersection (Möller–Trumbore), 1 ray per thread");
}


In [ ]:
%%writefile step1_forward.py
"""
Step 1 — Deterministic forward pass on CUDA.

Loads the .cu kernel via torch.utils.cpp_extension.load (JIT, no setup.py),
then validates it against the CPU reference from step 0 on:
  (a) the 12 hand-verifiable cases
  (b) the edge-biased sampling distribution (N = 1e5 by default, scale to 1e6 for the full check)

Run on any CUDA-equipped machine (laptop GPU, lab box, Colab). macOS will fail at load_extension.
"""

from __future__ import annotations

import os
import sys
import time

import numpy as np
import torch

from step0_cpu_reference import (
    ray_triangle_intersect,
    edge_distance_bary,
    edge_biased_sample,
)

THIS_DIR = os.path.dirname(os.path.abspath(__file__))


def _load_extension():
    from torch.utils.cpp_extension import load
    return load(
        name="diffsoup_step1",
        sources=[os.path.join(THIS_DIR, "step1_cuda_forward.cu")],
        verbose=True,
    )


def cuda_intersect(ext, origins_np, dirs_np, V0, V1, V2, dtype=torch.float32):
    origins = torch.as_tensor(origins_np, dtype=dtype, device="cuda")
    dirs    = torch.as_tensor(dirs_np,    dtype=dtype, device="cuda")
    v0 = torch.as_tensor(V0, dtype=dtype, device="cuda")
    v1 = torch.as_tensor(V1, dtype=dtype, device="cuda")
    v2 = torch.as_tensor(V2, dtype=dtype, device="cuda")
    hit, t, bary = ext.intersect_forward(origins, dirs, v0, v1, v2)
    torch.cuda.synchronize()
    return hit.cpu().numpy(), t.cpu().numpy(), bary.cpu().numpy()


# ---------------------------------------------------------------------------
# (a) 12 hand-verifiable cases — same as step 0
# ---------------------------------------------------------------------------

def test_hand_cases(ext) -> bool:
    V0 = np.array([0.0, 0.0, 0.0])
    V1 = np.array([1.0, 0.0, 0.0])
    V2 = np.array([0.0, 1.0, 0.0])

    cases = [
        ("centroid hit",           [1/3, 1/3, +1.0],           [0, 0, -1], True,  [1/3, 1/3, 1/3]),
        ("near vertex V0",         [0.0, 0.0, +1.0],           [0, 0, -1], True,  [1.0, 0.0, 0.0]),
        ("near vertex V1",         [1.0, 0.0, +1.0],           [0, 0, -1], True,  [0.0, 1.0, 0.0]),
        ("near vertex V2",         [0.0, 1.0, +1.0],           [0, 0, -1], True,  [0.0, 0.0, 1.0]),
        ("edge midpoint V1V2",     [0.5, 0.5, +1.0],           [0, 0, -1], True,  [0.0, 0.5, 0.5]),
        ("just inside edge V1V2",  [0.495, 0.495, +1.0],       [0, 0, -1], True,  [0.010, 0.495, 0.495]),
        ("just outside edge V1V2", [0.51, 0.51, +1.0],         [0, 0, -1], False, None),
        ("outside (x<0)",          [-0.01, 0.5, +1.0],         [0, 0, -1], False, None),
        ("outside (y<0)",          [0.5, -0.01, +1.0],         [0, 0, -1], False, None),
        ("parallel to plane",      [0.3, 0.3, +1.0],           [1, 0,  0], False, None),
        ("ray points away",        [0.3, 0.3, +1.0],           [0, 0, +1], False, None),
        ("oblique hit (centroid)", [1/3 + 1.0, 1/3 + 1.0, 1.0], [-1, -1, -1], True, [1/3, 1/3, 1/3]),
    ]
    origins = np.array([c[1] for c in cases], dtype=np.float64)
    dirs    = np.array([c[2] for c in cases], dtype=np.float64)

    hit_g, t_g, bary_g = cuda_intersect(ext, origins, dirs, V0, V1, V2, dtype=torch.float32)

    header = f"{'#':>2}  {'case':<26} {'hit':>5}  {'t':>9}  {'bary (w,u,v)':<28}  result"
    print(header)
    print("-" * len(header))

    all_ok = True
    for i, (name, _, _, exp_hit, exp_bary) in enumerate(cases):
        ok = bool(hit_g[i]) == exp_hit
        if exp_hit and exp_bary is not None:
            ok = ok and np.allclose(bary_g[i], exp_bary, atol=1e-5)
        t_str = f"{t_g[i]:9.5f}" if hit_g[i] else "    —    "
        bs = "[" + ", ".join(f"{c:+.4f}" if not np.isnan(c) else "  nan " for c in bary_g[i]) + "]"
        flag = "OK " if ok else "FAIL"
        all_ok = all_ok and ok
        print(f"{i:>2}  {name:<26} {str(bool(hit_g[i])):>5}  {t_str}  {bs:<28}  {flag}")

    print()
    print(("ALL 12 CASES PASSED" if all_ok else "SOME CASES FAILED") + " (CUDA)")
    return all_ok


# ---------------------------------------------------------------------------
# (b) Edge-biased sampling vs CPU reference
# ---------------------------------------------------------------------------

def test_against_cpu(ext, N: int = 100_000, seed: int = 0,
                     atol_bary: float = 1e-5, atol_t: float = 1e-4) -> bool:
    V0 = np.array([0.0, 0.0, 0.0])
    V1 = np.array([1.0, 0.0, 0.0])
    V2 = np.array([0.0, 1.0, 0.0])

    origins, dirs = edge_biased_sample(N, seed=seed)

    # CPU ground truth (float64)
    hit_c, t_c, bary_c = ray_triangle_intersect(origins, dirs, V0, V1, V2)

    # GPU under test (float32 — kernel default)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    hit_g, t_g, bary_g = cuda_intersect(ext, origins, dirs, V0, V1, V2, dtype=torch.float32)
    elapsed_ms = (time.perf_counter() - t0) * 1000.0

    print(f"\nedge-biased sampling: N={N}  (GPU intersect + copy back: {elapsed_ms:.1f} ms)")

    hit_match = np.array_equal(hit_c, hit_g)
    print(f"  hit flags match exactly      : {hit_match}")

    # Where both agree it's a hit, compare numerics.
    both_hit = hit_c & hit_g
    nb = int(both_hit.sum())
    if nb == 0:
        print("  no overlapping hits to compare — abort")
        return False

    bary_err = np.abs(bary_c[both_hit] - bary_g[both_hit])
    t_err = np.abs(t_c[both_hit] - t_g[both_hit])
    bary_ok = bary_err.max() < atol_bary
    t_ok    = t_err.max()    < atol_t

    print(f"  bary max abs err (N={nb})    : {bary_err.max():.3e}   (tol {atol_bary:.0e})  {'OK' if bary_ok else 'FAIL'}")
    print(f"  t    max abs err             : {t_err.max():.3e}   (tol {atol_t:.0e})  {'OK' if t_ok else 'FAIL'}")

    edge_d = edge_distance_bary(bary_g[hit_g])
    print(f"  GPU hit rate                 : {hit_g.mean():.4f}")
    print(f"  GPU mean min(bary)           : {edge_d.mean():.5f}   (smaller = nearer edges)")
    print(f"  GPU fraction within 0.02     : {(edge_d < 0.02).mean():.4f}")

    return hit_match and bary_ok and t_ok


# ---------------------------------------------------------------------------
# Entry
# ---------------------------------------------------------------------------

def main() -> int:
    if not torch.cuda.is_available():
        print("CUDA is not available in this environment — step 1 requires an NVIDIA GPU.", file=sys.stderr)
        print(f"  torch={torch.__version__}  platform={sys.platform}", file=sys.stderr)
        return 2

    print(f"torch {torch.__version__}  /  device: {torch.cuda.get_device_name(0)}")
    print("compiling step1_cuda_forward.cu ...")
    ext = _load_extension()
    print("compiled. running tests.\n")

    ok1 = test_hand_cases(ext)
    # Bump N to 1_000_000 for the full plan-spec sweep; 100k keeps iteration fast.
    N = int(os.environ.get("DIFFSOUP_N", "100000"))
    ok2 = test_against_cpu(ext, N=N)

    print()
    if ok1 and ok2:
        print("STEP 1 OK — CUDA forward pass matches CPU reference within tolerance.")
        return 0
    print("STEP 1 FAILED — see above.")
    return 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile step2_cuda_opacity.cu
// Step 2 — Smooth opacity probability α via a window function I(p). Still deterministic
// (no stochastic decision yet — that lands in step 3). 1 ray per thread.
//
// Idea: convert the ray's in-plane hit point into a SIGNED PERPENDICULAR DISTANCE d to the
// nearest triangle edge (world units, +inside / -outside), then
//     α = I(p) = sigmoid(d / σ).
// On the edge d=0 -> α=0.5; deep inside d≫σ -> α→1; outside d≪-σ -> α→0. The transition
// band has width ~σ, so σ=0.01 gives a thin soft edge.
//
// Crucially we compute α even when the ray geometrically MISSES the triangle (as long as it
// hits the plane in front): rays straddling the edge are exactly where the gradient lives in
// later steps, so we must not NaN them out here.
//
// Output convention (matches step0/step1 for bary):
//   bary  = (w, u, v) with P = w*V0 + u*V1 + v*V2,  w = 1 - u - v
//   sdist = signed perpendicular distance to nearest edge (world units)
//   alpha = sigmoid(sdist / sigma)  in [0, 1]
//   ray misses the plane (parallel) or hits behind origin -> alpha = 0, sdist/bary = NaN

#include <torch/extension.h>
#include <cuda_runtime.h>
#include <vector>

// Numerically stable logistic sigmoid.
template <typename scalar_t>
__device__ __forceinline__ scalar_t stable_sigmoid(scalar_t x) {
    if (x >= (scalar_t)0) {
        const scalar_t z = exp(-x);
        return (scalar_t)1 / ((scalar_t)1 + z);
    } else {
        const scalar_t z = exp(x);
        return z / ((scalar_t)1 + z);
    }
}

template <typename scalar_t>
__global__ void opacity_kernel(
    const scalar_t* __restrict__ origins,     // (N, 3) row-major
    const scalar_t* __restrict__ directions,  // (N, 3) row-major
    const scalar_t* __restrict__ verts,       // 9 floats: v0xyz, v1xyz, v2xyz
    scalar_t        sigma,
    scalar_t*       __restrict__ alpha,        // (N,)
    scalar_t*       __restrict__ sdist,        // (N,)
    scalar_t*       __restrict__ bary,         // (N, 3)
    int N)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    const scalar_t EPS = (scalar_t)1e-8;
    const scalar_t NaN = (scalar_t)NAN;

    const scalar_t v0x = verts[0], v0y = verts[1], v0z = verts[2];
    const scalar_t v1x = verts[3], v1y = verts[4], v1z = verts[5];
    const scalar_t v2x = verts[6], v2y = verts[7], v2z = verts[8];

    const scalar_t ox = origins[3*i+0], oy = origins[3*i+1], oz = origins[3*i+2];
    const scalar_t dx = directions[3*i+0], dy = directions[3*i+1], dz = directions[3*i+2];

    const scalar_t e1x = v1x - v0x, e1y = v1y - v0y, e1z = v1z - v0z;
    const scalar_t e2x = v2x - v0x, e2y = v2y - v0y, e2z = v2z - v0z;

    // --- Möller–Trumbore: barycentric (u, v, w) and parametric t ---
    const scalar_t px = dy*e2z - dz*e2y;
    const scalar_t py = dz*e2x - dx*e2z;
    const scalar_t pz = dx*e2y - dy*e2x;

    const scalar_t det = px*e1x + py*e1y + pz*e1z;
    const bool parallel = (det > -EPS) && (det < EPS);
    const scalar_t inv_det = parallel ? (scalar_t)0 : (scalar_t)1 / det;

    const scalar_t tx = ox - v0x, ty = oy - v0y, tz = oz - v0z;
    const scalar_t u = (tx*px + ty*py + tz*pz) * inv_det;

    const scalar_t qx = ty*e1z - tz*e1y;
    const scalar_t qy = tz*e1x - tx*e1z;
    const scalar_t qz = tx*e1y - ty*e1x;

    const scalar_t v = (dx*qx + dy*qy + dz*qz) * inv_det;
    const scalar_t t = (qx*e2x + qy*e2y + qz*e2z) * inv_det;
    const scalar_t w = (scalar_t)1 - u - v;

    // Only require an in-front plane intersection. We do NOT require u,v,w >= 0:
    // points just outside the triangle still get a (small) opacity so the edge is smooth.
    const bool valid = (!parallel) && (t > EPS);

    if (!valid) {
        alpha[i]    = (scalar_t)0;
        sdist[i]    = NaN;
        bary[3*i+0] = NaN; bary[3*i+1] = NaN; bary[3*i+2] = NaN;
        return;
    }

    // --- barycentric -> signed perpendicular distance to each edge (world units) ---
    // |n| = 2 * Area, where n = e1 x e2. Altitude from vertex i = |n| / L_i, with L_i the
    // length of the edge opposite vertex i. Perp distance to that edge = bary_i * altitude_i.
    const scalar_t nx = e1y*e2z - e1z*e2y;
    const scalar_t ny = e1z*e2x - e1x*e2z;
    const scalar_t nz = e1x*e2y - e1y*e2x;
    const scalar_t two_area = sqrt(nx*nx + ny*ny + nz*nz);   // |n|

    // Edge opposite V0 is V1V2; opposite V1 is V0V2 (=e2); opposite V2 is V0V1 (=e1).
    const scalar_t l0x = v2x - v1x, l0y = v2y - v1y, l0z = v2z - v1z;
    const scalar_t L0 = sqrt(l0x*l0x + l0y*l0y + l0z*l0z);
    const scalar_t L1 = sqrt(e2x*e2x + e2y*e2y + e2z*e2z);
    const scalar_t L2 = sqrt(e1x*e1x + e1y*e1y + e1z*e1z);

    const scalar_t d0 = w * (two_area / L0);   // distance to edge opposite V0
    const scalar_t d1 = u * (two_area / L1);   // distance to edge opposite V1
    const scalar_t d2 = v * (two_area / L2);   // distance to edge opposite V2

    scalar_t d = d0 < d1 ? d0 : d1;
    d = d < d2 ? d : d2;                        // signed distance to nearest edge

    alpha[i]    = stable_sigmoid(d / sigma);
    sdist[i]    = d;
    bary[3*i+0] = w;
    bary[3*i+1] = u;
    bary[3*i+2] = v;
}

std::vector<torch::Tensor> opacity_forward(
    torch::Tensor origins,
    torch::Tensor directions,
    torch::Tensor v0,
    torch::Tensor v1,
    torch::Tensor v2,
    double sigma)
{
    TORCH_CHECK(origins.is_cuda(),    "origins must be CUDA");
    TORCH_CHECK(directions.is_cuda(), "directions must be CUDA");
    TORCH_CHECK(origins.dim() == 2 && origins.size(1) == 3,    "origins must be (N, 3)");
    TORCH_CHECK(directions.sizes() == origins.sizes(),         "directions must match origins");
    TORCH_CHECK(origins.scalar_type() == directions.scalar_type(), "dtype mismatch");
    TORCH_CHECK(v0.numel() == 3 && v1.numel() == 3 && v2.numel() == 3, "vertices must have 3 elements");
    TORCH_CHECK(sigma > 0.0, "sigma must be positive");

    origins    = origins.contiguous();
    directions = directions.contiguous();

    const auto N = origins.size(0);
    const auto opts = origins.options();

    auto verts = torch::empty({9}, opts);
    verts.narrow(0, 0, 3).copy_(v0.to(opts.device()).to(opts.dtype()).view({3}));
    verts.narrow(0, 3, 3).copy_(v1.to(opts.device()).to(opts.dtype()).view({3}));
    verts.narrow(0, 6, 3).copy_(v2.to(opts.device()).to(opts.dtype()).view({3}));

    auto alpha = torch::empty({N},    opts);
    auto sdist = torch::empty({N},    opts);
    auto bary  = torch::empty({N, 3}, opts);

    const int threads = 256;
    const int blocks  = (int)((N + threads - 1) / threads);

    AT_DISPATCH_FLOATING_TYPES(origins.scalar_type(), "opacity_kernel", [&] {
        opacity_kernel<scalar_t><<<blocks, threads>>>(
            origins.data_ptr<scalar_t>(),
            directions.data_ptr<scalar_t>(),
            verts.data_ptr<scalar_t>(),
            (scalar_t)sigma,
            alpha.data_ptr<scalar_t>(),
            sdist.data_ptr<scalar_t>(),
            bary.data_ptr<scalar_t>(),
            (int)N);
    });

    return {alpha, sdist, bary};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("opacity_forward", &opacity_forward,
          "Smooth opacity α = sigmoid(signed_edge_distance / sigma), 1 ray per thread");
}

In [ ]:
%%writefile step2_opacity.py
"""
Step 2 — Smooth opacity probability α (deterministic, no sampling yet).

The CUDA kernel turns each ray's in-plane hit point into a signed perpendicular
distance d to the nearest triangle edge, then α = sigmoid(d / σ).

Validates the kernel against a NumPy reference on:
  (a) a perpendicular sweep across an edge  — α must follow sigmoid(d/σ) and the
      10–90% transition width must be ln(81)·σ ≈ 4.394·σ
  (b) the edge-biased sampling distribution  — elementwise α match vs CPU (N≈1e5)

Run on any CUDA machine (Colab T4 is fine). macOS fails at load_extension.
"""

from __future__ import annotations

import math
import os
import sys
import time

import numpy as np
import torch

from step0_cpu_reference import ray_triangle_intersect, edge_biased_sample

THIS_DIR = os.path.dirname(os.path.abspath(__file__))
SIGMA = 0.01

V0 = np.array([0.0, 0.0, 0.0])
V1 = np.array([1.0, 0.0, 0.0])
V2 = np.array([0.0, 1.0, 0.0])


def _load_extension():
    from torch.utils.cpp_extension import load
    return load(
        name="diffsoup_step2",
        sources=[os.path.join(THIS_DIR, "step2_cuda_opacity.cu")],
        verbose=True,
    )


# ---------------------------------------------------------------------------
# CPU reference for the opacity window — mirrors the kernel exactly.
# ---------------------------------------------------------------------------

def opacity_reference(origins, dirs, v0, v1, v2, sigma):
    """Returns (alpha, sdist, bary) with the same convention as the kernel.

    Misses the plane (parallel / behind) -> alpha 0, sdist & bary NaN.
    """
    origins = np.asarray(origins, dtype=np.float64)
    dirs = np.asarray(dirs, dtype=np.float64)

    hit_plane, t, bary = _intersect_plane(origins, dirs, v0, v1, v2)

    w, u, v = bary[:, 0], bary[:, 1], bary[:, 2]

    # |n| = 2*Area, altitude_i = |n| / L_i, d_i = bary_i * altitude_i
    n = np.cross(v1 - v0, v2 - v0)
    two_area = np.linalg.norm(n)
    L0 = np.linalg.norm(v2 - v1)     # edge opposite V0
    L1 = np.linalg.norm(v2 - v0)     # edge opposite V1
    L2 = np.linalg.norm(v1 - v0)     # edge opposite V2

    d0 = w * (two_area / L0)
    d1 = u * (two_area / L1)
    d2 = v * (two_area / L2)
    d = np.minimum(np.minimum(d0, d1), d2)

    alpha = _sigmoid(d / sigma)

    nan = np.full(origins.shape[0], np.nan)
    alpha = np.where(hit_plane, alpha, 0.0)
    sdist = np.where(hit_plane, d, nan)
    bary_out = np.where(hit_plane[:, None], bary, np.nan)
    return alpha, sdist, bary_out


def _intersect_plane(origins, dirs, v0, v1, v2, eps=1e-8):
    """Like step0 but does NOT clamp to inside the triangle; only needs t>eps."""
    e1, e2 = v1 - v0, v2 - v0
    pvec = np.cross(dirs, e2)
    det = pvec @ e1
    parallel = np.abs(det) < eps
    inv_det = np.where(parallel, 0.0, 1.0 / np.where(parallel, 1.0, det))
    tvec = origins - v0
    u = np.einsum("ij,ij->i", tvec, pvec) * inv_det
    qvec = np.cross(tvec, e1)
    v = np.einsum("ij,ij->i", dirs, qvec) * inv_det
    t = (qvec @ e2) * inv_det
    w = 1.0 - u - v
    hit_plane = (~parallel) & (t > eps)
    bary = np.stack([w, u, v], axis=-1)
    return hit_plane, t, bary


def _sigmoid(x):
    return np.where(x >= 0, 1.0 / (1.0 + np.exp(-x)), np.exp(x) / (1.0 + np.exp(x)))


# ---------------------------------------------------------------------------
# CUDA call
# ---------------------------------------------------------------------------

def cuda_opacity(ext, origins_np, dirs_np, sigma, dtype=torch.float32):
    origins = torch.as_tensor(origins_np, dtype=dtype, device="cuda")
    dirs = torch.as_tensor(dirs_np, dtype=dtype, device="cuda")
    v0 = torch.as_tensor(V0, dtype=dtype, device="cuda")
    v1 = torch.as_tensor(V1, dtype=dtype, device="cuda")
    v2 = torch.as_tensor(V2, dtype=dtype, device="cuda")
    alpha, sdist, bary = ext.opacity_forward(origins, dirs, v0, v1, v2, float(sigma))
    torch.cuda.synchronize()
    return alpha.cpu().numpy(), sdist.cpu().numpy(), bary.cpu().numpy()


# ---------------------------------------------------------------------------
# (a) Perpendicular sweep across edge V0V1 (the x-axis edge, y = 0).
#     A point at (x0, y, 0) inside has perpendicular distance d = y to that edge.
#     Sweep y from -5σ to +5σ; α should trace sigmoid(y/σ).
# ---------------------------------------------------------------------------

def test_sweep(ext, sigma=SIGMA, n=801):
    x0 = 0.3                                   # safely between the two other edges
    ys = np.linspace(-5 * sigma, 5 * sigma, n)
    pts = np.stack([np.full(n, x0), ys, np.zeros(n)], axis=-1)
    origins = pts + np.array([0.0, 0.0, 1.0])
    dirs = np.tile(np.array([0.0, 0.0, -1.0]), (n, 1))

    alpha_g, sdist_g, _ = cuda_opacity(ext, origins, dirs, sigma)
    alpha_ref = _sigmoid(ys / sigma)            # exact: d == y on this sweep

    max_err = np.nanmax(np.abs(alpha_g - alpha_ref))

    # Empirical 10–90% transition width should equal ln(81)·σ.
    expected_width = math.log(81.0) * sigma
    y_lo = np.interp(0.10, alpha_g, ys)
    y_hi = np.interp(0.90, alpha_g, ys)
    width = y_hi - y_lo
    width_err = abs(width - expected_width)

    a_center = float(np.interp(0.0, ys, alpha_g))
    monotone = bool(np.all(np.diff(alpha_g) >= -1e-7))

    print(f"\n(a) perpendicular sweep across edge V0V1  (σ={sigma}, N={n})")
    print(f"  α vs sigmoid(y/σ) max abs err : {max_err:.3e}   (tol 2e-3)  "
          f"{'OK' if max_err < 2e-3 else 'FAIL'}")
    print(f"  α on the edge (y=0)           : {a_center:.5f}   (expect 0.5)")
    print(f"  α deep inside (+5σ)           : {alpha_g[-1]:.5f}   α outside (−5σ): {alpha_g[0]:.5f}")
    print(f"  10–90% band width             : {width:.5f}   expect {expected_width:.5f}  "
          f"(err {width_err:.2e})  {'OK' if width_err < 0.1 * sigma else 'FAIL'}")
    print(f"  monotonically increasing      : {monotone}")

    ok = (max_err < 2e-3) and (abs(a_center - 0.5) < 1e-2) \
        and (width_err < 0.1 * sigma) and monotone
    return ok


# ---------------------------------------------------------------------------
# (b) Edge-biased distribution: elementwise α match vs CPU reference.
# ---------------------------------------------------------------------------

def test_against_cpu(ext, sigma=SIGMA, N=100_000, seed=0, atol=2e-3):
    origins, dirs = edge_biased_sample(N, seed=seed)

    alpha_c, sdist_c, _ = opacity_reference(origins, dirs, V0, V1, V2, sigma)

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    alpha_g, sdist_g, _ = cuda_opacity(ext, origins, dirs, sigma)
    elapsed = (time.perf_counter() - t0) * 1000.0

    a_err = np.abs(alpha_c - alpha_g)
    s_err = np.abs(np.nan_to_num(sdist_c) - np.nan_to_num(sdist_g))
    a_ok = a_err.max() < atol
    s_ok = s_err.max() < 1e-4

    print(f"\n(b) edge-biased dist vs CPU  (N={N}, GPU {elapsed:.1f} ms)")
    print(f"  α    max abs err : {a_err.max():.3e}   (tol {atol:.0e})  {'OK' if a_ok else 'FAIL'}")
    print(f"  sdist max abs err: {s_err.max():.3e}   (tol 1e-4)  {'OK' if s_ok else 'FAIL'}")
    print(f"  α mean           : {alpha_g.mean():.4f}   frac(α<0.5 → near/outside edge): "
          f"{(alpha_g < 0.5).mean():.4f}")
    return a_ok and s_ok


def main() -> int:
    if not torch.cuda.is_available():
        print("CUDA is not available — step 2 requires an NVIDIA GPU.", file=sys.stderr)
        print(f"  torch={torch.__version__}  platform={sys.platform}", file=sys.stderr)
        return 2

    print(f"torch {torch.__version__}  /  device: {torch.cuda.get_device_name(0)}")
    print("compiling step2_cuda_opacity.cu ...")
    ext = _load_extension()
    print("compiled. running tests.")

    ok_a = test_sweep(ext)
    N = int(os.environ.get("DIFFSOUP_N", "100000"))
    ok_b = test_against_cpu(ext, N=N)

    print()
    if ok_a and ok_b:
        print("STEP 2 OK — opacity window is smooth, σ-calibrated, and matches CPU reference.")
        return 0
    print("STEP 2 FAILED — see above.")
    return 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile step3_cuda_stochastic.cu
// Step 3 — Stochastic opacity decision. Builds on step 2's smooth α, then turns it into a
// Bernoulli sample: draw ξ ~ U(0,1] per ray and decide opaque ⇔ (ξ < α). 1 ray per thread.
//
// Why stochastic? A hard inside/outside test is a step function — ∂/∂V = 0 almost everywhere,
// so the edge carries no gradient. Replacing it with a coin flip whose bias is α makes the
// *expected* contribution equal to α, which is smooth in V. Averaging M samples at a fixed
// point converges to α (Monte Carlo); the gradient of that expectation is the score-function
// (REINFORCE) estimator we wire up in step 4. This step just produces the samples + α.
//
// RNG: cuRAND Philox (counter-based). Seed is shared; the per-thread *sequence* = ray index,
// so the 10^6 streams are independent and the whole launch is reproducible for a given seed.
//
// Output:
//   alpha  (N,)  smooth opacity probability in [0,1]   (same as step 2)
//   sample (N,)  Bernoulli draw in {0.0, 1.0}: 1 = opaque/blocked, 0 = transmitted
//   ray misses the plane (parallel / behind) -> alpha = 0, sample = 0

#include <torch/extension.h>
#include <cuda_runtime.h>
#include <curand_kernel.h>
#include <vector>

template <typename scalar_t>
__device__ __forceinline__ scalar_t stable_sigmoid(scalar_t x) {
    if (x >= (scalar_t)0) {
        const scalar_t z = exp(-x);
        return (scalar_t)1 / ((scalar_t)1 + z);
    } else {
        const scalar_t z = exp(x);
        return z / ((scalar_t)1 + z);
    }
}

template <typename scalar_t>
__global__ void stochastic_kernel(
    const scalar_t* __restrict__ origins,     // (N, 3) row-major
    const scalar_t* __restrict__ directions,  // (N, 3) row-major
    const scalar_t* __restrict__ verts,       // 9 floats: v0xyz, v1xyz, v2xyz
    scalar_t          sigma,
    unsigned long long seed,
    scalar_t*       __restrict__ alpha,        // (N,)
    scalar_t*       __restrict__ sample,       // (N,)  {0,1}
    int N)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    const scalar_t EPS = (scalar_t)1e-8;

    const scalar_t v0x = verts[0], v0y = verts[1], v0z = verts[2];
    const scalar_t v1x = verts[3], v1y = verts[4], v1z = verts[5];
    const scalar_t v2x = verts[6], v2y = verts[7], v2z = verts[8];

    const scalar_t ox = origins[3*i+0], oy = origins[3*i+1], oz = origins[3*i+2];
    const scalar_t dx = directions[3*i+0], dy = directions[3*i+1], dz = directions[3*i+2];

    const scalar_t e1x = v1x - v0x, e1y = v1y - v0y, e1z = v1z - v0z;
    const scalar_t e2x = v2x - v0x, e2y = v2y - v0y, e2z = v2z - v0z;

    // --- Möller–Trumbore: barycentric (u, v, w), parametric t ---
    const scalar_t px = dy*e2z - dz*e2y;
    const scalar_t py = dz*e2x - dx*e2z;
    const scalar_t pz = dx*e2y - dy*e2x;

    const scalar_t det = px*e1x + py*e1y + pz*e1z;
    const bool parallel = (det > -EPS) && (det < EPS);
    const scalar_t inv_det = parallel ? (scalar_t)0 : (scalar_t)1 / det;

    const scalar_t tx = ox - v0x, ty = oy - v0y, tz = oz - v0z;
    const scalar_t u = (tx*px + ty*py + tz*pz) * inv_det;

    const scalar_t qx = ty*e1z - tz*e1y;
    const scalar_t qy = tz*e1x - tx*e1z;
    const scalar_t qz = tx*e1y - ty*e1x;

    const scalar_t v = (dx*qx + dy*qy + dz*qz) * inv_det;
    const scalar_t t = (qx*e2x + qy*e2y + qz*e2z) * inv_det;
    const scalar_t w = (scalar_t)1 - u - v;

    const bool valid = (!parallel) && (t > EPS);

    scalar_t a = (scalar_t)0;
    if (valid) {
        // barycentric -> signed perpendicular distance to nearest edge (world units)
        const scalar_t nx = e1y*e2z - e1z*e2y;
        const scalar_t ny = e1z*e2x - e1x*e2z;
        const scalar_t nz = e1x*e2y - e1y*e2x;
        const scalar_t two_area = sqrt(nx*nx + ny*ny + nz*nz);

        const scalar_t l0x = v2x - v1x, l0y = v2y - v1y, l0z = v2z - v1z;
        const scalar_t L0 = sqrt(l0x*l0x + l0y*l0y + l0z*l0z);
        const scalar_t L1 = sqrt(e2x*e2x + e2y*e2y + e2z*e2z);
        const scalar_t L2 = sqrt(e1x*e1x + e1y*e1y + e1z*e1z);

        const scalar_t d0 = w * (two_area / L0);
        const scalar_t d1 = u * (two_area / L1);
        const scalar_t d2 = v * (two_area / L2);
        scalar_t d = d0 < d1 ? d0 : d1;
        d = d < d2 ? d : d2;

        a = stable_sigmoid(d / sigma);
    }

    // --- stochastic decision: ξ ~ U(0,1], opaque ⇔ ξ < α -------------------
    // Philox: shared seed, per-ray sequence = i -> independent reproducible streams.
    curandStatePhilox4_32_10_t state;
    curand_init(seed, (unsigned long long)i, 0ULL, &state);
    const float xi = curand_uniform(&state);          // (0, 1]
    const scalar_t s = (xi < (float)a) ? (scalar_t)1 : (scalar_t)0;

    alpha[i]  = a;
    sample[i] = valid ? s : (scalar_t)0;
}

std::vector<torch::Tensor> stochastic_forward(
    torch::Tensor origins,
    torch::Tensor directions,
    torch::Tensor v0,
    torch::Tensor v1,
    torch::Tensor v2,
    double sigma,
    int64_t seed)
{
    TORCH_CHECK(origins.is_cuda(),    "origins must be CUDA");
    TORCH_CHECK(directions.is_cuda(), "directions must be CUDA");
    TORCH_CHECK(origins.dim() == 2 && origins.size(1) == 3, "origins must be (N, 3)");
    TORCH_CHECK(directions.sizes() == origins.sizes(),      "directions must match origins");
    TORCH_CHECK(origins.scalar_type() == directions.scalar_type(), "dtype mismatch");
    TORCH_CHECK(v0.numel() == 3 && v1.numel() == 3 && v2.numel() == 3, "vertices must have 3 elements");
    TORCH_CHECK(sigma > 0.0, "sigma must be positive");

    origins    = origins.contiguous();
    directions = directions.contiguous();

    const auto N = origins.size(0);
    const auto opts = origins.options();

    auto verts = torch::empty({9}, opts);
    verts.narrow(0, 0, 3).copy_(v0.to(opts.device()).to(opts.dtype()).view({3}));
    verts.narrow(0, 3, 3).copy_(v1.to(opts.device()).to(opts.dtype()).view({3}));
    verts.narrow(0, 6, 3).copy_(v2.to(opts.device()).to(opts.dtype()).view({3}));

    auto alpha  = torch::empty({N}, opts);
    auto sample = torch::empty({N}, opts);

    const int threads = 256;
    const int blocks  = (int)((N + threads - 1) / threads);

    AT_DISPATCH_FLOATING_TYPES(origins.scalar_type(), "stochastic_kernel", [&] {
        stochastic_kernel<scalar_t><<<blocks, threads>>>(
            origins.data_ptr<scalar_t>(),
            directions.data_ptr<scalar_t>(),
            verts.data_ptr<scalar_t>(),
            (scalar_t)sigma,
            (unsigned long long)seed,
            alpha.data_ptr<scalar_t>(),
            sample.data_ptr<scalar_t>(),
            (int)N);
    });

    return {alpha, sample};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("stochastic_forward", &stochastic_forward,
          "Stochastic opacity: Bernoulli(α) per ray via Philox RNG, 1 ray per thread");
}


In [ ]:
%%writefile step3_stochastic.py
"""
Step 3 — Stochastic opacity decision.

Each ray draws ξ ~ U(0,1] (Philox) and emits a Bernoulli sample opaque ⇔ (ξ < α).
The mean over many samples at a fixed point must converge to α — that is the whole
point: E[sample] = α is smooth in V, so it carries a usable gradient (step 4), while a
hard inside/outside test would not.

Validates the kernel on:
  (a) convergence  — replicate a point M times; |mean − α| within Monte-Carlo error,
                     for several target α values
  (b) reproducibility — same seed ⇒ bit-identical samples; different seed ⇒ differs,
                        but the mean is unchanged within MC error (= no race / correct streams)
  (c) smoothness   — sweep across an edge, binned mean traces sigmoid(d/σ); full 10^6 sweep

Run on any CUDA machine (Colab T4). macOS fails at load_extension.
"""

from __future__ import annotations

import math
import os
import sys
import time

import numpy as np
import torch

from step0_cpu_reference import edge_biased_sample

THIS_DIR = os.path.dirname(os.path.abspath(__file__))
SIGMA = 0.01

V0 = np.array([0.0, 0.0, 0.0])
V1 = np.array([1.0, 0.0, 0.0])
V2 = np.array([0.0, 1.0, 0.0])


def _load_extension():
    from torch.utils.cpp_extension import load
    return load(
        name="diffsoup_step3",
        sources=[os.path.join(THIS_DIR, "step3_cuda_stochastic.cu")],
        verbose=True,
    )


def cuda_stochastic(ext, origins_np, dirs_np, sigma, seed, dtype=torch.float32):
    origins = torch.as_tensor(origins_np, dtype=dtype, device="cuda")
    dirs = torch.as_tensor(dirs_np, dtype=dtype, device="cuda")
    v0 = torch.as_tensor(V0, dtype=dtype, device="cuda")
    v1 = torch.as_tensor(V1, dtype=dtype, device="cuda")
    v2 = torch.as_tensor(V2, dtype=dtype, device="cuda")
    alpha, sample = ext.stochastic_forward(origins, dirs, v0, v1, v2, float(sigma), int(seed))
    torch.cuda.synchronize()
    return alpha.cpu().numpy(), sample.cpu().numpy()


def _rays_at_perp_distance(d_values, x0=0.3):
    """Rays that hit edge V0V1 (y=0) at signed perpendicular distance d (= y here)."""
    d_values = np.asarray(d_values, dtype=np.float64)
    pts = np.stack([np.full(d_values.shape, x0), d_values, np.zeros_like(d_values)], axis=-1)
    origins = pts + np.array([0.0, 0.0, 1.0])
    dirs = np.tile(np.array([0.0, 0.0, -1.0]), (d_values.shape[0], 1))
    return origins, dirs


# ---------------------------------------------------------------------------
# (a) convergence of the sample mean to α
# ---------------------------------------------------------------------------

def test_convergence(ext, sigma=SIGMA, M=200_000, seed=1234):
    targets = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
    # d such that sigmoid(d/σ) = target
    d = sigma * np.log(targets / (1.0 - targets))

    print(f"\n(a) convergence  (M={M} samples per target, σ={sigma})")
    print(f"  {'target α':>9} {'kernel α':>9} {'mean':>9} {'|mean−α|':>10} {'4σ_MC':>9}  result")
    all_ok = True
    for tgt, dd in zip(targets, d):
        origins, dirs = _rays_at_perp_distance(np.full(M, dd))
        alpha_g, sample_g = cuda_stochastic(ext, origins, dirs, sigma, seed)
        a = float(alpha_g[0])
        mean = float(sample_g.mean())
        mc = 4.0 * math.sqrt(a * (1.0 - a) / M)   # ~4 std-errors band
        ok = abs(mean - a) < mc
        all_ok = all_ok and ok
        print(f"  {tgt:9.2f} {a:9.4f} {mean:9.4f} {abs(mean-a):10.2e} {mc:9.2e}  "
              f"{'OK' if ok else 'FAIL'}")
    return all_ok


# ---------------------------------------------------------------------------
# (b) reproducibility / independent streams
# ---------------------------------------------------------------------------

def test_reproducibility(ext, sigma=SIGMA, M=200_000):
    origins, dirs = _rays_at_perp_distance(np.zeros(M))   # all on the edge, α=0.5
    a1, s1 = cuda_stochastic(ext, origins, dirs, sigma, seed=7)
    a2, s2 = cuda_stochastic(ext, origins, dirs, sigma, seed=7)    # same seed
    a3, s3 = cuda_stochastic(ext, origins, dirs, sigma, seed=99)   # different seed

    same_seed_identical = np.array_equal(s1, s2)
    frac_diff = float((s1 != s3).mean())
    # independent fair coins differ ~50% of the time
    streams_ok = 0.45 < frac_diff < 0.55
    means = (s1.mean(), s3.mean())
    means_ok = abs(means[0] - 0.5) < 5e-3 and abs(means[1] - 0.5) < 5e-3

    print(f"\n(b) reproducibility  (M={M} rays on the edge, α=0.5)")
    print(f"  same seed → bit-identical samples : {same_seed_identical}  "
          f"{'OK' if same_seed_identical else 'FAIL'}")
    print(f"  diff seed → fraction differing     : {frac_diff:.4f}   (expect ~0.50)  "
          f"{'OK' if streams_ok else 'FAIL'}")
    print(f"  means (seed7, seed99)              : {means[0]:.4f}, {means[1]:.4f}   "
          f"{'OK' if means_ok else 'FAIL'}")
    return same_seed_identical and streams_ok and means_ok


# ---------------------------------------------------------------------------
# (c) smoothness — binned sample mean across the edge follows sigmoid(d/σ)
# ---------------------------------------------------------------------------

def test_smoothness(ext, sigma=SIGMA, N=1_000_000, seed=2024):
    # Spread N rays uniformly in y over [-5σ, 5σ] crossing edge V0V1.
    ys = np.linspace(-5 * sigma, 5 * sigma, N)
    origins, dirs = _rays_at_perp_distance(ys)

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    alpha_g, sample_g = cuda_stochastic(ext, origins, dirs, sigma, seed)
    elapsed = (time.perf_counter() - t0) * 1000.0

    # Bin by y and compare bin mean vs sigmoid(y/σ).
    nb = 40
    edges = np.linspace(-5 * sigma, 5 * sigma, nb + 1)
    idx = np.clip(np.digitize(ys, edges) - 1, 0, nb - 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    counts = np.bincount(idx, minlength=nb)
    sums = np.bincount(idx, weights=sample_g, minlength=nb)
    bin_mean = sums / np.maximum(counts, 1)
    bin_ref = 1.0 / (1.0 + np.exp(-centers / sigma))

    valid = counts > 100
    rmse = float(np.sqrt(np.mean((bin_mean[valid] - bin_ref[valid]) ** 2)))
    # monotone (allowing MC wiggle)
    smoothed = bin_mean[valid]
    monotone_ish = float(np.mean(np.diff(smoothed) >= -0.05)) > 0.9

    print(f"\n(c) smoothness sweep  (N={N}, {nb} bins, GPU {elapsed:.1f} ms)")
    print(f"  binned mean vs sigmoid  RMSE : {rmse:.4f}   (tol 0.02)  "
          f"{'OK' if rmse < 0.02 else 'FAIL'}")
    print(f"  bin mean @ edge (y≈0)        : "
          f"{bin_mean[nb//2]:.3f} / {bin_mean[nb//2-1]:.3f}   (expect ~0.5)")
    print(f"  overall sample rate          : {sample_g.mean():.4f}   "
          f"mean α: {alpha_g.mean():.4f}  (should match within MC)")
    print(f"  roughly monotone increasing  : {monotone_ish}")
    return (rmse < 0.02) and monotone_ish


def main() -> int:
    if not torch.cuda.is_available():
        print("CUDA is not available — step 3 requires an NVIDIA GPU.", file=sys.stderr)
        print(f"  torch={torch.__version__}  platform={sys.platform}", file=sys.stderr)
        return 2

    print(f"torch {torch.__version__}  /  device: {torch.cuda.get_device_name(0)}")
    print("compiling step3_cuda_stochastic.cu ...")
    ext = _load_extension()
    print("compiled. running tests.")

    ok_a = test_convergence(ext)
    ok_b = test_reproducibility(ext)
    N = int(os.environ.get("DIFFSOUP_N", "1000000"))
    ok_c = test_smoothness(ext, N=N)

    print()
    if ok_a and ok_b and ok_c:
        print("STEP 3 OK — stochastic samples converge to α, streams are independent & reproducible.")
        return 0
    print("STEP 3 FAILED — see above.")
    return 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile step4_cuda_backward.cu
// Step 4 — Backward pass with atomics.
//
// Gradient of the smooth opacity α w.r.t. the (in-plane) vertex positions, accumulated into a
// shared 9-float buffer with atomicAdd. This is the DiffSoup trick: the FORWARD render is a
// stochastic Bernoulli(α) sample (step 3), but the GRADIENT flows through the smooth, exact
// expectation E[sample] = α(V), which is differentiable. So
//     dL/dV = Σ_i (dL/dα_i) · (dα_i/dV)            (pathwise, low-variance)
// is what we accumulate. (The score-function / REINFORCE estimator would target the same
// expected gradient with higher variance; we use the smooth-α path here.)
//
// Signed distance to the edge opposite vertex k, with the hit point P fixed in the plane:
//     d_k = ((Va - P) × (Vb - Va)) / |Vb - Va|        (× = 2D scalar cross, CCW triangle)
// where (Va,Vb) is that edge oriented CCW. This equals λ_k·|n|/L_k from steps 2–3 but depends
// only on the edge's two endpoints, so dα_i/dV touches exactly the two nearest-edge vertices —
// the opposite vertex gets zero. α = sigmoid(min_k d_k / σ),  dα/dd = α(1-α)/σ.
//
// All N rays write the SAME 3 vertices ⇒ heavy contention on 6 floats. atomicAdd serializes the
// writes so there is no race (the sum is correct); FP add-ordering still varies run-to-run, which
// is NOT a race (step 6 distinguishes the two).
//
// Scope: vertices lie in z = 0 and rays shoot along ±z (the challenge test case). Gradients are
// computed for the x,y of each vertex; the z column stays 0 (in-plane perturbations keep P fixed).

#include <torch/extension.h>
#include <cuda_runtime.h>
#include <vector>

template <typename scalar_t>
__device__ __forceinline__ scalar_t stable_sigmoid(scalar_t x) {
    if (x >= (scalar_t)0) { const scalar_t z = exp(-x); return (scalar_t)1 / ((scalar_t)1 + z); }
    else                  { const scalar_t z = exp(x);  return z / ((scalar_t)1 + z); }
}

// 2D signed distance from P to the directed line A->B. Positive on the left (CCW) side.
template <typename scalar_t>
__device__ __forceinline__ scalar_t sdist2d(
    scalar_t px, scalar_t py, scalar_t ax, scalar_t ay, scalar_t bx, scalar_t by)
{
    const scalar_t ex = bx - ax, ey = by - ay;
    const scalar_t L  = sqrt(ex*ex + ey*ey);
    const scalar_t cr = (ax - px) * ey - (ay - py) * ex;
    return cr / L;
}

// Möller–Trumbore -> (valid, hit point P). Shared by forward and backward so both agree exactly.
template <typename scalar_t>
__device__ __forceinline__ bool hit_point(
    const scalar_t* o, const scalar_t* d, const scalar_t* V,
    scalar_t& px, scalar_t& py, scalar_t& pz)
{
    const scalar_t EPS = (scalar_t)1e-8;
    const scalar_t e1x = V[3]-V[0], e1y = V[4]-V[1], e1z = V[5]-V[2];
    const scalar_t e2x = V[6]-V[0], e2y = V[7]-V[1], e2z = V[8]-V[2];
    const scalar_t hx = d[1]*e2z - d[2]*e2y;
    const scalar_t hy = d[2]*e2x - d[0]*e2z;
    const scalar_t hz = d[0]*e2y - d[1]*e2x;
    const scalar_t det = hx*e1x + hy*e1y + hz*e1z;
    if (det > -EPS && det < EPS) return false;
    const scalar_t inv = (scalar_t)1 / det;
    const scalar_t tx = o[0]-V[0], ty = o[1]-V[1], tz = o[2]-V[2];
    const scalar_t qx = ty*e1z - tz*e1y, qy = tz*e1x - tx*e1z, qz = tx*e1y - ty*e1x;
    const scalar_t t  = (qx*e2x + qy*e2y + qz*e2z) * inv;
    if (t <= EPS) return false;
    px = o[0] + t*d[0]; py = o[1] + t*d[1]; pz = o[2] + t*d[2];
    return true;
}

// Nearest edge: returns min signed distance and the edge index (0,1,2 opposite V0,V1,V2).
template <typename scalar_t>
__device__ __forceinline__ scalar_t nearest_edge(
    scalar_t px, scalar_t py, const scalar_t* V, int& edge)
{
    // edge opposite V0 -> (V1,V2); opposite V1 -> (V2,V0); opposite V2 -> (V0,V1)
    const scalar_t d0 = sdist2d(px,py, V[3],V[4], V[6],V[7]);
    const scalar_t d1 = sdist2d(px,py, V[6],V[7], V[0],V[1]);
    const scalar_t d2 = sdist2d(px,py, V[0],V[1], V[3],V[4]);
    scalar_t d = d0; edge = 0;
    if (d1 < d) { d = d1; edge = 1; }
    if (d2 < d) { d = d2; edge = 2; }
    return d;
}

// ---------------------------------------------------------------------------
// Forward: α = sigmoid(min edge distance / σ).  Matches steps 2–3.
// ---------------------------------------------------------------------------
template <typename scalar_t>
__global__ void forward_kernel(
    const scalar_t* __restrict__ origins, const scalar_t* __restrict__ directions,
    const scalar_t* __restrict__ verts, scalar_t sigma,
    scalar_t* __restrict__ alpha, int N)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    scalar_t px, py, pz;
    if (!hit_point(origins + 3*i, directions + 3*i, verts, px, py, pz)) { alpha[i] = 0; return; }
    int edge; const scalar_t d = nearest_edge(px, py, verts, edge);
    alpha[i] = stable_sigmoid(d / sigma);
}

// ---------------------------------------------------------------------------
// Backward: accumulate dL/dV = Σ_i grad_alpha_i · dα_i/dV  via atomicAdd.
// ---------------------------------------------------------------------------
template <typename scalar_t>
__global__ void backward_kernel(
    const scalar_t* __restrict__ origins, const scalar_t* __restrict__ directions,
    const scalar_t* __restrict__ verts, scalar_t sigma,
    const scalar_t* __restrict__ grad_alpha,   // (N,) upstream dL/dα
    scalar_t* __restrict__ grad_verts,         // (9,) accumulated
    int N)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    scalar_t px, py, pz;
    if (!hit_point(origins + 3*i, directions + 3*i, verts, px, py, pz)) return;

    int edge;
    const scalar_t d = nearest_edge(px, py, verts, edge);
    const scalar_t a = stable_sigmoid(d / sigma);
    const scalar_t dadd = a * ((scalar_t)1 - a) / sigma;   // dα/dd
    const scalar_t factor = grad_alpha[i] * dadd;          // dL/dα · dα/dd = dL/dd

    // Endpoints (A,B) of the active edge, oriented CCW, and their vertex slots.
    int ia, ib;
    if      (edge == 0) { ia = 1; ib = 2; }   // (V1,V2)
    else if (edge == 1) { ia = 2; ib = 0; }   // (V2,V0)
    else                { ia = 0; ib = 1; }   // (V0,V1)

    const scalar_t ax = verts[3*ia+0], ay = verts[3*ia+1];
    const scalar_t bx = verts[3*ib+0], by = verts[3*ib+1];
    const scalar_t ex = bx - ax, ey = by - ay;
    const scalar_t L2 = ex*ex + ey*ey;
    const scalar_t L  = sqrt(L2);
    const scalar_t cr = (ax - px) * ey - (ay - py) * ex;   // = d * L

    // ∂d/∂(endpoint coord) = ∂cross/∂·/L − cross·∂L/∂·/L²
    const scalar_t dd_ax = (ey + (ay - py)) / L - cr * (-ex / L) / L2;
    const scalar_t dd_ay = (-(ax - px) - ex) / L - cr * (-ey / L) / L2;
    const scalar_t dd_bx = (-(ay - py))     / L - cr * ( ex / L) / L2;
    const scalar_t dd_by = ( (ax - px))     / L - cr * ( ey / L) / L2;

    atomicAdd(&grad_verts[3*ia+0], factor * dd_ax);
    atomicAdd(&grad_verts[3*ia+1], factor * dd_ay);
    atomicAdd(&grad_verts[3*ib+0], factor * dd_bx);
    atomicAdd(&grad_verts[3*ib+1], factor * dd_by);
    // z components (slots 2,5,8) are untouched: in-plane gradient only.
}

// ---------------------------------------------------------------------------
// Host wrappers
// ---------------------------------------------------------------------------
static torch::Tensor pack_verts(torch::Tensor v0, torch::Tensor v1, torch::Tensor v2,
                                const torch::TensorOptions& opts) {
    auto verts = torch::empty({9}, opts);
    verts.narrow(0,0,3).copy_(v0.to(opts.device()).to(opts.dtype()).view({3}));
    verts.narrow(0,3,3).copy_(v1.to(opts.device()).to(opts.dtype()).view({3}));
    verts.narrow(0,6,3).copy_(v2.to(opts.device()).to(opts.dtype()).view({3}));
    return verts;
}

torch::Tensor opacity_forward(torch::Tensor origins, torch::Tensor directions,
                              torch::Tensor v0, torch::Tensor v1, torch::Tensor v2, double sigma) {
    TORCH_CHECK(origins.is_cuda() && directions.is_cuda(), "inputs must be CUDA");
    TORCH_CHECK(origins.dim()==2 && origins.size(1)==3, "origins must be (N,3)");
    origins = origins.contiguous(); directions = directions.contiguous();
    const auto N = origins.size(0); const auto opts = origins.options();
    auto verts = pack_verts(v0,v1,v2,opts);
    auto alpha = torch::empty({N}, opts);
    const int threads = 256, blocks = (int)((N+threads-1)/threads);
    AT_DISPATCH_FLOATING_TYPES(origins.scalar_type(), "forward_kernel", [&]{
        forward_kernel<scalar_t><<<blocks,threads>>>(
            origins.data_ptr<scalar_t>(), directions.data_ptr<scalar_t>(),
            verts.data_ptr<scalar_t>(), (scalar_t)sigma, alpha.data_ptr<scalar_t>(), (int)N);
    });
    return alpha;
}

torch::Tensor opacity_backward(torch::Tensor origins, torch::Tensor directions,
                               torch::Tensor v0, torch::Tensor v1, torch::Tensor v2,
                               double sigma, torch::Tensor grad_alpha) {
    TORCH_CHECK(origins.is_cuda() && directions.is_cuda() && grad_alpha.is_cuda(), "inputs must be CUDA");
    TORCH_CHECK(grad_alpha.numel()==origins.size(0), "grad_alpha must be (N,)");
    origins = origins.contiguous(); directions = directions.contiguous();
    grad_alpha = grad_alpha.contiguous();
    const auto N = origins.size(0); const auto opts = origins.options();
    auto verts = pack_verts(v0,v1,v2,opts);
    auto grad_verts = torch::zeros({9}, opts);          // <- zero-init; kernel accumulates
    const int threads = 256, blocks = (int)((N+threads-1)/threads);
    AT_DISPATCH_FLOATING_TYPES(origins.scalar_type(), "backward_kernel", [&]{
        backward_kernel<scalar_t><<<blocks,threads>>>(
            origins.data_ptr<scalar_t>(), directions.data_ptr<scalar_t>(),
            verts.data_ptr<scalar_t>(), (scalar_t)sigma,
            grad_alpha.data_ptr<scalar_t>(), grad_verts.data_ptr<scalar_t>(), (int)N);
    });
    return grad_verts.view({3,3});
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("opacity_forward",  &opacity_forward,  "Smooth opacity α (2D signed-distance form)");
    m.def("opacity_backward", &opacity_backward, "Accumulate dL/dV via atomicAdd (in-plane)");
}


In [ ]:
%%writefile step4_backward.py
"""
Steps 4–6 — Backward pass, gradient correctness, and race-condition check.

Step 4: opacity_backward accumulates dL/dV = Σ_i grad_alpha_i · dα_i/dV into a 9-float buffer
        with atomicAdd (all N rays hit the same 3 vertices ⇒ maximal contention).

Step 5 (gradient check): with L = Σ_i α_i(V), compare the kernel's analytic ∂L/∂V against
        central finite differences of the GPU forward. Done in float64 so there is NO
        Monte-Carlo noise (gradient flows through the smooth α, not the stochastic sample),
        and FD/analytic should agree to ~1e-5 relative.

Step 6 (race check): run the float32 backward K times on identical input. atomicAdd makes the
        sum correct but FP add-ordering varies run to run — that jitter must be tiny and the
        mean must match a float64 reference. A real race would show large, unstable swings.

Run on any CUDA machine (Colab T4). macOS fails at load_extension.
"""

from __future__ import annotations

import os
import sys
import time

import numpy as np
import torch

from step0_cpu_reference import edge_biased_sample

THIS_DIR = os.path.dirname(os.path.abspath(__file__))
SIGMA = 0.01

V0 = [0.0, 0.0, 0.0]
V1 = [1.0, 0.0, 0.0]
V2 = [0.0, 1.0, 0.0]


def _load_extension():
    from torch.utils.cpp_extension import load
    return load(
        name="diffsoup_step4",
        sources=[os.path.join(THIS_DIR, "step4_cuda_backward.cu")],
        verbose=True,
    )


def _verts(dtype):
    return (torch.tensor(V0, dtype=dtype, device="cuda"),
            torch.tensor(V1, dtype=dtype, device="cuda"),
            torch.tensor(V2, dtype=dtype, device="cuda"))


# ---------------------------------------------------------------------------
# Step 5 — finite-difference gradient check (deterministic, float64)
# ---------------------------------------------------------------------------

def test_gradient(ext, N=200_000, sigma=SIGMA, eps=1e-6, seed=0):
    dtype = torch.float64
    origins_np, dirs_np = edge_biased_sample(N, seed=seed)
    origins = torch.as_tensor(origins_np, dtype=dtype, device="cuda")
    dirs = torch.as_tensor(dirs_np, dtype=dtype, device="cuda")
    v0, v1, v2 = _verts(dtype)
    verts = [v0, v1, v2]

    def S(vs):
        return ext.opacity_forward(origins, dirs, vs[0], vs[1], vs[2], float(sigma)).sum().item()

    # analytic dL/dV with L = Σ α  ->  grad_alpha = 1
    grad_alpha = torch.ones(N, dtype=dtype, device="cuda")
    analytic = ext.opacity_backward(origins, dirs, v0, v1, v2, float(sigma), grad_alpha)
    analytic = analytic.cpu().numpy()           # (3,3)

    fd = np.zeros((3, 3))
    for vi in range(3):
        for c in range(2):                       # x,y only (in-plane); z grad is 0 by construction
            base = verts[vi].clone()
            vp = base.clone(); vp[c] += eps
            vm = base.clone(); vm[c] -= eps
            vs_p = list(verts); vs_p[vi] = vp
            vs_m = list(verts); vs_m[vi] = vm
            fd[vi, c] = (S(vs_p) - S(vs_m)) / (2 * eps)

    scale = max(np.abs(fd).max(), 1e-12)
    abs_err = np.abs(analytic[:, :2] - fd[:, :2])
    rel = abs_err.max() / scale
    z_zero = np.abs(analytic[:, 2]).max()

    print(f"\n(step 5) gradient check  (N={N}, float64, ε={eps:g})")
    print(f"  L = Σ_i α_i,  grad shape (3 verts × xyz)")
    names = ["V0", "V1", "V2"]
    print(f"  {'':4}{'analytic ∂L/∂x':>16}{'FD ∂L/∂x':>14}{'analytic ∂L/∂y':>16}{'FD ∂L/∂y':>14}")
    for vi in range(3):
        print(f"  {names[vi]:4}{analytic[vi,0]:16.4f}{fd[vi,0]:14.4f}"
              f"{analytic[vi,1]:16.4f}{fd[vi,1]:14.4f}")
    print(f"  max abs err (x,y)         : {abs_err.max():.3e}")
    print(f"  max relative err          : {rel:.3e}   (tol 1e-4)  {'OK' if rel < 1e-4 else 'FAIL'}")
    print(f"  z-gradient (expect 0)     : {z_zero:.3e}  {'OK' if z_zero < 1e-12 else 'FAIL'}")
    return (rel < 1e-4) and (z_zero < 1e-12)


# ---------------------------------------------------------------------------
# Step 6 — race / determinism check (float32 atomics)
# ---------------------------------------------------------------------------

def test_race(ext, N=1_000_000, sigma=SIGMA, K=50, seed=0):
    origins_np, dirs_np = edge_biased_sample(N, seed=seed)

    # float64 reference (atomic FP jitter ~1e-12 → effectively ground truth)
    o64 = torch.as_tensor(origins_np, dtype=torch.float64, device="cuda")
    d64 = torch.as_tensor(dirs_np, dtype=torch.float64, device="cuda")
    v0d, v1d, v2d = _verts(torch.float64)
    g64 = ext.opacity_backward(o64, d64, v0d, v1d, v2d, float(sigma),
                               torch.ones(N, dtype=torch.float64, device="cuda")).cpu().numpy()

    o32 = torch.as_tensor(origins_np, dtype=torch.float32, device="cuda")
    d32 = torch.as_tensor(dirs_np, dtype=torch.float32, device="cuda")
    v0f, v1f, v2f = _verts(torch.float32)
    ones32 = torch.ones(N, dtype=torch.float32, device="cuda")

    runs = []
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(K):
        g = ext.opacity_backward(o32, d32, v0f, v1f, v2f, float(sigma), ones32)
        torch.cuda.synchronize()
        runs.append(g.cpu().numpy())
    elapsed = (time.perf_counter() - t0) * 1000.0 / K
    runs = np.stack(runs)                         # (K, 3, 3)

    spread = runs.max(0) - runs.min(0)            # run-to-run swing per component
    mag = np.abs(g64) + 1e-8
    rel_spread = (spread / mag).max()
    rel_bias = (np.abs(runs.mean(0) - g64) / mag).max()
    bit_exact = bool(np.all(runs.max(0) == runs.min(0)))

    print(f"\n(step 6) race / determinism  (N={N}, {K} runs, float32, {elapsed:.1f} ms/run)")
    print(f"  run-to-run spread (rel)   : {rel_spread:.3e}")
    print(f"  bias vs float64 ref (rel) : {rel_bias:.3e}")
    print(f"  bit-exact across runs     : {bit_exact}  "
          f"(False is normal — FP add-ordering, not a race)")
    # No race ⇔ jitter is at FP-ordering scale (small) AND mean matches the f64 reference.
    no_race = (rel_spread < 1e-3) and (rel_bias < 1e-3)
    print(f"  verdict                   : {'NO RACE (jitter is FP-ordering only)' if no_race else 'POSSIBLE RACE — investigate'}"
          f"  {'OK' if no_race else 'FAIL'}")
    return no_race


def main() -> int:
    if not torch.cuda.is_available():
        print("CUDA is not available — steps 4–6 require an NVIDIA GPU.", file=sys.stderr)
        print(f"  torch={torch.__version__}  platform={sys.platform}", file=sys.stderr)
        return 2

    print(f"torch {torch.__version__}  /  device: {torch.cuda.get_device_name(0)}")
    print("compiling step4_cuda_backward.cu ...")
    ext = _load_extension()
    print("compiled. running tests.")

    ok5 = test_gradient(ext)
    N = int(os.environ.get("DIFFSOUP_N", "1000000"))
    ok6 = test_race(ext, N=N)

    print()
    if ok5 and ok6:
        print("STEPS 4–6 OK — gradient matches finite differences and atomics are race-free.")
        return 0
    print("STEPS 4–6 FAILED — see above.")
    return 1


if __name__ == "__main__":
    sys.exit(main())


## 3. Sanity check — CPU reference

In [ ]:
!python step0_cpu_reference.py


## 4. Step 1 — JIT-compile the CUDA kernel and validate against CPU

First run compiles `step1_cuda_forward.cu` (≈30–60 s); subsequent runs are cached.
Set `DIFFSOUP_N=1000000` to do the full plan-spec sweep (1e6 rays).


In [ ]:
!DIFFSOUP_N=100000 python step1_forward.py


### Optional — full 10^6 ray sweep

In [ ]:
!DIFFSOUP_N=1000000 python step1_forward.py


## 5. Step 2 — Smooth opacity window α = sigmoid(d / σ)

Sweeps a ray perpendicular across an edge: α must fall smoothly from ~1 (inside) through 0.5 (on the edge) to ~0 (outside), with a 10–90% band width of ln(81)·σ ≈ 4.39σ for σ=0.01. Then checks elementwise α against the CPU reference.


In [ ]:
!DIFFSOUP_N=100000 python step2_opacity.py


## 6. Step 3 — Stochastic decision (Bernoulli sampling + Philox RNG)

Each ray draws ξ ~ U(0,1] and emits opaque ⇔ (ξ < α). Checks that the sample mean converges to α, that the same seed is bit-reproducible while different seeds give independent streams (~50% differ), and that a full 10^6-ray sweep across the edge traces sigmoid(d/σ). Runs the full 1e6 sweep by default.


In [ ]:
!python step3_stochastic.py


## 7. Steps 4–6 — Backward pass, gradient check, race check

The gradient flows through the smooth α (E[sample]=α), so it carries a usable, low-variance signal. Step 5 confirms the kernel's analytic ∂L/∂V matches central finite differences in float64 (no Monte-Carlo noise). Step 6 runs the float32 backward 50× on identical input: atomicAdd keeps the sum correct, so the run-to-run swing should sit at FP-ordering scale (not a race) and the mean must match the float64 reference.


In [ ]:
!python step4_backward.py
